# Stats over Corpus for Word Chapter

In [ ]:
#first parse the corpus
# Import the toolkit, giving it a shorter alias 'mptf'
import mptf_parser as mptf

# Set folder paths
input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output"

In [ ]:
import os
from collections import defaultdict
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf

# ======================================================
# SETTINGS & PATHS
# ======================================================

input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output"

os.makedirs(output_folder, exist_ok=True)

# ======================================================
# STEP 1: CREATE CONLLU FILES (ONCE)
# ======================================================

print("--- Starting Corpus Parsing Pipeline ---")
mptf._process_directory(input_folder, output_folder)

# ======================================================
# STEP 2: LOAD CONLLU FILES INTO CORPUS
# ======================================================

print(f"\n--- Loading files from '{output_folder}' ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
)

for filename in conllu_files:
    # 🚫 SKIP OUTDATED FILES
    if "outdated" in filename.lower():
        continue

    file_path = os.path.join(output_folder, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    # --- Add to corpus ---
    for sent in sentences:
        sentence_obj = mptf.Sentence(
            sent,
            source_filename=filename
        )
        sentence_obj.metadata["source_filename"] = filename
        my_corpus.append(sentence_obj)

print(f"✔ Corpus created with {len(my_corpus)} sentences.")

# ======================================================
# STEP 3: PER-FILE STATISTICS
# ======================================================

print("\n--- Generating Per-File Statistics ---")

file_stats = defaultdict(lambda: {
    "sentence_count": 0,
    "tokens": []
})

for sentence in my_corpus:
    fname = sentence.metadata["source_filename"]
    file_stats[fname]["sentence_count"] += 1

    for token in sentence.get_tokens():
        form = str(token.form).strip()
        if form:
            file_stats[fname]["tokens"].append(form)

# ======================================================
# STEP 4: PRINT TABLE
# ======================================================

header = (
    f"{'Filename':<30} | "
    f"{'Sents':<6} | "
    f"{'Tokens':<8} | "
    f"{'Types':<8} | "
    f"{'MWEs':<6}"
)

print("\n" + header)
print("-" * len(header))

for filename, stats in sorted(file_stats.items()):
    tokens = stats["tokens"]
    sents = stats["sentence_count"]

    types = set(tokens)
    mwes = [t for t in types if " " in t or "|" in t]

    print(
        f"{filename[:30]:<30} | "
        f"{sents:<6} | "
        f"{len(tokens):<8} | "
        f"{len(types):<8} | "
        f"{len(mwes):<6}"
    )

# ======================================================
# STEP 5: GLOBAL SUMMARY
# ======================================================

all_tokens = [
    tok
    for stats in file_stats.values()
    for tok in stats["tokens"]
]

print("\n--- Global Corpus Summary ---")
print(f"Total sentences: {len(my_corpus)}")
print(f"Total tokens:    {len(all_tokens)}")
print(f"Total types:     {len(set(all_tokens))}")


--- Starting Corpus Parsing Pipeline ---


Converting source files:   0%|          | 0/46 [00:00<?, ?it/s]

In [ ]:
# ======================================================
# PERCENTAGE OF DK7-B UP TO 2.52
# ======================================================

target_file = "Dk7-B"

# Step 1: Extract all DK7-B sentences
all_dk7_sentences = [
    s for s in my_corpus
    if target_file in s.metadata.get("source_filename", "")
]

# Step 2: Count sentences/tokens/types up to 2.52 (reuse previous logic)
max_part = 2.52
dk7_sentences = []
dk7_tokens = []

for sentence in all_dk7_sentences:

    sentence_part = None

    for token in sentence.get_tokens():
        misc = getattr(token, "misc", None)
        if not misc:
            continue

        for key in misc.keys():
            if key.startswith("Dk7 "):
                try:
                    number_str = key.replace("Dk7", "").strip()
                    sentence_part = float(number_str)
                    break
                except ValueError:
                    pass

        if sentence_part is not None:
            break

    if sentence_part is not None and sentence_part <= max_part:
        dk7_sentences.append(sentence)
        dk7_tokens.extend(
            str(t.form).strip()
            for t in sentence.get_tokens()
            if str(t.form).strip()
        )

# Step 3: Compute percentages
total_sentences = len(all_dk7_sentences)
total_tokens = sum(len(s.get_tokens()) for s in all_dk7_sentences)

sent_pct = (len(dk7_sentences) / total_sentences) * 100 if total_sentences else 0
token_pct = (len(dk7_tokens) / total_tokens) * 100 if total_tokens else 0

print("\n--- DK7-B up to Dk7 2.52: Percentage of Entire DK7-B ---")
print(f"Sentences: {len(dk7_sentences)} / {total_sentences} ({sent_pct:.2f}%)")
print(f"Tokens:    {len(dk7_tokens)} / {total_tokens} ({token_pct:.2f}%)")
print(f"Types:     {len(set(dk7_tokens))}")


--- DK7-B up to Dk7 2.52: Percentage of Entire DK7-B ---
Sentences: 255 / 626 (40.73%)
Tokens:    3962 / 18997 (20.86%)
Types:     907


In [ ]:
from collections import Counter

# --- STEP 1: Flatten all tokens from corpus ---
all_tokens = []
for sentence in my_corpus:
    for token in sentence.get_tokens():
        token_form = str(token.form).strip()
        if token_form:  # ignore empty
            all_tokens.append(token_form)

print(f"Total tokens (all occurrences): {len(all_tokens)}")

# --- STEP 2: Count distinct lemmata ---
distinct_lemmata = set(all_tokens)
print(f"Distinct lemmata (types): {len(distinct_lemmata)}")

# --- STEP 3: Identify multiword expressions (MWEs) ---
MWEs = [lemma for lemma in distinct_lemmata if " " in lemma]
print(f"Multiword expressions (MWEs): {len(MWEs)}")

# Optional: show first 10 MWEs
print("\nExamples of MWEs:")
for mwe in MWEs[:10]:
    print(" ", mwe)

# --- STEP 4 (Optional): Count total occurrences of MWEs ---
mwe_counts = Counter(token for token in all_tokens if "|" in token)
print(f"\nTotal MWE occurrences in corpus: {sum(mwe_counts.values())}")


Total tokens (all occurrences): 1388292
Distinct lemmata (types): 30932
Multiword expressions (MWEs): 189

Examples of MWEs:
  rārišn xxx
  yōhē pasčēta
  xxx widard [without be?]
  čāšīdār yyy
  zād xxx
  ānābišn xxx
  dād xxx
  ahlaw xxx
  ōšmurēd xxx
  im xxxx

Total MWE occurrences in corpus: 6


--- Starting Corpus Parsing Pipeline ---


Converting source files:   0%|          | 0/44 [00:00<?, ?it/s]


Source file conversion complete!

Step 2: Loading files from 'C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output' and generating my_corpus.
Successfully created 'my_corpus' with 77405 total sentences.

--- Generating Per-File Statistics ---

Filename                            | Genre        | Sents  | Tokens   | Types    | MWEs  
------------------------------------------------------------------------------------------
AOD-K20_mptf.conllu                 | unknown      | 212    | 3463     | 487      | 0     
AWN.conllu                          | unknown      | 730    | 10933    | 1218     | 0     
AfZ-TD2_mptf.conllu                 | zand         | 90     | 688      | 367      | 0     
Col-J2-1-J2_mptf.conllu             | colophon     | 5      | 124      | 69       | 0     
Col-J2_mptf.conllu                  | unknown      | 7      | 114      | 63       | 0     
Col-K1-1-K1_mptf.conllu             | colophon     | 24     | 487      | 206      | 0 

--- Starting Corpus Parsing Pipeline ---


Converting source files:   0%|          | 0/44 [00:00<?, ?it/s]


Source file conversion complete!

--- Loading files from 'C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output' ---
✔ Corpus created with 31191 sentences.

--- Generating Per-File Statistics ---

Filename                       | Sents  | Tokens   | Types    | MWEs  
----------------------------------------------------------------------
AOD-K20_mptf.conllu            | 106    | 1732     | 486      | 0     
Col-J2_mptf.conllu             | 7      | 114      | 63       | 0     
Col-K20-1_mptf.conllu          | 3      | 50       | 38       | 0     
Col-K43a-2_mptf.conllu         | 5      | 86       | 54       | 0     
Col-K5-1_mptf.conllu           | 9      | 133      | 100      | 0     
Col-K7b-1_mptf.conllu          | 2      | 17       | 17       | 0     
Col-K7b-2_mptf.conllu          | 9      | 154      | 99       | 0     
Col-TD1-1_mptf.conllu          | 3      | 80       | 48       | 0     
Col-TD2-1_mptf.conllu          | 4      | 53       | 42      

In [ ]:
import csv
import os

# --- CSV output path ---
csv_file_path = os.path.join(output_folder, "corpus_statistics.csv")

# Column headers
fields = [
    'Filename',
    'Genre',
    'Sentences',
    'Tokens',
    'Types',
    'TTR (%)',
    'Token Ratio vs Corpus (%)',   # <-- New column
    'Relative TTR vs Mean Text (%)'  # Option B
]

print(f"\nExporting results to: {csv_file_path}")

# ==================================================
# STEP 1: COMPUTE GLOBAL CORPUS TOKEN COUNT
# ==================================================
all_tokens_global = [t for stats in file_stats.values() for t in stats["tokens"]]
global_token_count = len(all_tokens_global)
print(f"Global corpus token count: {global_token_count}")

# ==================================================
# STEP 2: COMPUTE PER-TEXT TTRs AND MEAN TTR
# ==================================================
text_ttrs = {}

for filename, stats in file_stats.items():
    tokens = stats["tokens"]
    num_tokens = len(tokens)
    num_types = len(set(tokens))
    ttr = (num_types / num_tokens) if num_tokens > 0 else 0
    text_ttrs[filename] = ttr

mean_text_ttr = sum(text_ttrs.values()) / len(text_ttrs) if text_ttrs else 0
print(f"Mean TTR across texts: {mean_text_ttr:.4f}")

# ==================================================
# STEP 3: WRITE CSV
# ==================================================
with open(csv_file_path, mode='w', encoding='utf-8-sig', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()

    for filename, stats in sorted(file_stats.items()):
        tokens = stats["tokens"]
        num_tokens = len(tokens)
        num_types = len(set(tokens))
        ttr = text_ttrs[filename]

        # Token ratio vs corpus
        token_ratio_vs_corpus = (num_tokens / global_token_count * 100) if global_token_count > 0 else 0

        # Relative TTR vs mean text
        relative_vs_mean = (ttr / mean_text_ttr * 100) if mean_text_ttr > 0 else 0

        writer.writerow({
            'Filename': filename,
            
            'Sentences': stats["sentence_count"],
            'Tokens': num_tokens,
            'Types': num_types,
            'TTR (%)': round(ttr * 100, 2),
            'Token Ratio vs Corpus (%)': round(token_ratio_vs_corpus, 2),
            'Relative TTR vs Mean Text (%)': round(relative_vs_mean, 2)
        })

print("Export complete! now open 'corpus_statistics.csv' in Excel.")


NameError: name 'output_folder' is not defined

In [ ]:
#upos stats

In [ ]:
from collections import Counter

upos_counts = Counter()
total_tokens = 0

# Track which sentence IDs we've already counted
seen_sentence_ids = set()

for sentence in my_corpus:
    # Get the sentence ID
    sentence_id = sentence.metadata.get("SENTENCE ID", "UNKNOWN_SENTENCE_ID")
    
    # --- Choose one format only ---
    # Example: only keep IDs with the "DD-K35-01_sentence_" format
    if "sentence_" not in sentence_id:
        continue  # skip the other format

    # Skip duplicates
    if sentence_id in seen_sentence_ids:
        continue
    seen_sentence_ids.add(sentence_id)

    for token in sentence.get_tokens():
        # Skip placeholder tokens
        if not token.lemma or token.lemma in {"_", "__"}:
            continue

        upos = token.upos
        if upos:  # only count real UPOS
            upos_counts[upos] += 1
            total_tokens += 1

# Calculate percentages
upos_percentages = {upos: count / total_tokens * 100 for upos, count in upos_counts.items()}

# Display results
print("UPOS Counts:")
for upos, count in upos_counts.most_common():
    print(f"{upos}: {count}")

print("\nUPOS Percentages:")
for upos, percent in sorted(upos_percentages.items()):
    print(f"{upos}: {percent:.2f}%")


UPOS Counts:
NOUN: 2491
ADP: 1164
DET: 1112
VERB: 852
ADV: 803
PRON: 744
CCONJ: 726
ADJ: 520
SCONJ: 463
_: 457
PART: 271
AUX: 245
PROPN: 206
NUM: 171
CPD: 57

UPOS Percentages:
ADJ: 5.06%
ADP: 11.32%
ADV: 7.81%
AUX: 2.38%
CCONJ: 7.06%
CPD: 0.55%
DET: 10.82%
NOUN: 24.23%
NUM: 1.66%
PART: 2.64%
PRON: 7.24%
PROPN: 2.00%
SCONJ: 4.50%
VERB: 8.29%
_: 4.44%


In [ ]:
from collections import Counter, defaultdict

# -----------------------------
# Initialize counters
# -----------------------------
global_upos_counts = Counter()
total_global_tokens = 0

per_text_upos = defaultdict(Counter)
per_text_total_tokens = defaultdict(int)

# -----------------------------
# Iterate over corpus
# -----------------------------
for sentence in my_corpus:
    # Use source_filename as the text ID
    text_id = sentence.metadata.get('source_filename', 'unknown')

    for token in sentence.get_tokens():
        # Skip placeholder tokens
        if not token.lemma or token.lemma in {"_", "__"}:
            continue

        upos = token.upos
        if not upos:
            continue

        # Global counts
        global_upos_counts[upos] += 1
        total_global_tokens += 1

        # Per-text counts
        per_text_upos[text_id][upos] += 1
        per_text_total_tokens[text_id] += 1

# -----------------------------
# Compute ratios
# -----------------------------
# Global UPOS ratios
global_upos_ratios = {upos: count / total_global_tokens * 100
                      for upos, count in global_upos_counts.items()}

# Per-text UPOS ratios (relative to all tokens in that text)
per_text_upos_ratios = defaultdict(dict)
for text_id, counts in per_text_upos.items():
    total_tokens_in_text = per_text_total_tokens[text_id]
    for upos, count in counts.items():
        per_text_upos_ratios[text_id][upos] = (count / total_tokens_in_text * 100
                                               if total_tokens_in_text > 0 else 0)

# -----------------------------
# Display results
# -----------------------------
print("=== Global UPOS Ratios ===")
for upos, ratio in sorted(global_upos_ratios.items()):
    print(f"{upos}: {ratio:.2f}%")

print("\n=== Per-Text UPOS Ratios ===")
for text_id, ratios in sorted(per_text_upos_ratios.items()):
    print(f"\nText: {text_id}")
    for upos, ratio in sorted(ratios.items()):
        print(f"  {upos}: {ratio:.2f}%")


=== Global UPOS Ratios ===
ADJ: 5.92%
ADJV: 0.00%
ADJ|DET: 0.00%
ADP: 10.08%
ADP|ADP: 0.00%
ADV: 7.38%
ADV|PART: 0.00%
ADV|PRON: 0.00%
AUX: 2.19%
BELIEF: 0.00%
CCONJ: 7.09%
CPD: 0.40%
CPD|ADJ: 0.00%
DET: 10.81%
EZAFE: 0.00%
FIX: 0.00%
INTJ: 0.00%
NOUN: 24.95%
NOUN|ADJ: 0.00%
NOUN|PROPN: 0.00%
NUM: 1.88%
ON, OVER: 0.00%
PART: 1.80%
PRON: 6.65%
PROPN: 3.57%
PROPN|ADJ: 0.00%
PUNCT: 0.01%
SCONJ: 5.62%
VERB: 9.77%
X: 0.01%
YYY: 0.00%
ZZZ: 0.00%
_: 1.86%

=== Per-Text UPOS Ratios ===

Text: AOD-K20_mptf.conllu
  ADJ: 9.93%
  ADP: 9.33%
  ADV: 5.14%
  AUX: 2.81%
  CCONJ: 9.03%
  DET: 7.78%
  NOUN: 25.30%
  NUM: 3.71%
  PART: 3.23%
  PRON: 6.22%
  PROPN: 0.72%
  PUNCT: 0.06%
  SCONJ: 4.01%
  VERB: 11.24%
  X: 0.06%
  _: 1.44%

Text: Col-J2_mptf.conllu
  ADJ: 6.14%
  ADP: 9.65%
  ADV: 3.51%
  AUX: 0.88%
  CCONJ: 2.63%
  CPD: 0.88%
  DET: 9.65%
  NOUN: 21.93%
  NUM: 1.75%
  PRON: 5.26%
  PROPN: 6.14%
  SCONJ: 2.63%
  VERB: 8.77%
  _: 20.18%

Text: Col-K20-1_mptf.conllu
  ADJ: 2.08%
  ADP: 8.33%


In [ ]:
#investigating weird upos
from collections import defaultdict
input_upos = "NOUN|PROPN"  
# Dictionary to store results: {sentence_id: set of lemmas}
upos_sentences = defaultdict(set)

# --- Iterate over corpus ---
for sentence in my_corpus:
    sentence_id = sentence.metadata.get("SENTENCE ID", "UNKNOWN_SENTENCE_ID")
    
    for token in sentence.get_tokens():
        if token.upos == input_upos:   # input_upos is the UPOS searching for
            upos_sentences[sentence_id].add(token.lemma)

# --- Display results ---
for sent_id, lemmas in upos_sentences.items():
    print(f"Sentence ID: {sent_id}")
    print(f"Lemmas with UPOS '{input_upos}': {', '.join(sorted(lemmas))}\n")


Sentence ID: GBd-TD1 s1478	_	_	_	_	_	_	_	_	_
Lemmas with UPOS 'NOUN|PROPN': ādur



In [ ]:
#Feats investigation
from collections import Counter, defaultdict

# ======================================================
# DATA STRUCTURES
# ======================================================

# Total tokens per UPOS
upos_token_counts = Counter()

# Feature-key counts per UPOS
# {UPOS: Counter({Feature: count})}
upos_feat_key_counts = defaultdict(Counter)

# Feature-value counts per UPOS
# {UPOS: {Feature: Counter({Value: count})}}
upos_feat_value_counts = defaultdict(lambda: defaultdict(Counter))

# ======================================================
# ITERATE OVER CORPUS
# ======================================================

for sentence in my_corpus:
    for token in sentence.get_tokens():

        # Skip malformed / placeholder tokens
        if not token.upos or not token.lemma or token.lemma in {"_", "__"}:
            continue

        upos = token.upos
        feats = token.feats  # This is a dict or None

        # Count token per UPOS
        upos_token_counts[upos] += 1

        if not feats:
            continue

        # Iterate over FEATS dictionary
        for feat_key, feat_value in feats.items():
            if not feat_value:
                continue

            # Count feature-key occurrence
            upos_feat_key_counts[upos][feat_key] += 1

            # Handle multi-valued feats (e.g. Case=Nom|Acc)
            if isinstance(feat_value, list):
                for val in feat_value:
                    upos_feat_value_counts[upos][feat_key][val] += 1
            else:
                upos_feat_value_counts[upos][feat_key][feat_value] += 1

# ======================================================
# DISPLAY RESULTS
# ======================================================

for upos in sorted(upos_token_counts):
    print(f"\n==============================")
    print(f"UPOS: {upos}")
    print(f"Total tokens: {upos_token_counts[upos]}")
    print(f"------------------------------")

    # Feature keys
    print("Feature keys:")
    for feat, count in upos_feat_key_counts[upos].most_common():
        print(f"  {feat}: {count}")

    # Feature values
    print("\nFeature values:")
    for feat in sorted(upos_feat_value_counts[upos]):
        print(f"  {feat}:")
        for val, count in upos_feat_value_counts[upos][feat].most_common():
            print(f"    {val}: {count}")



UPOS: ADJ
Total tokens: 33342
------------------------------
Feature keys:
  Degree: 4422
  Number: 1285
  Typo: 780
  NumType: 761
  Subcat: 567
  VerbForm: 510
  Tense: 492
  Animacy: 152
  Transc: 53
  Gender: 50
  VerbType: 27
  Echo: 22
  AdvType: 16
  Foreign: 15
  Poss: 11
  Reflex: 11
  Person: 11
  PronType: 7
  Num: 7
  Mood: 6
  NameType: 4
  Case: 3
  Style: 1

Feature values:
  AdvType:
    Man: 9
    Tim: 4
    Deg: 3
  Animacy:
    Inan: 126
    Hum: 16
    Anim: 8
    Nhum: 2
  Case:
    Nom: 3
  Degree:
    Cmp: 3073
    Sup: 1347
    Abs: 2
  Echo:
    Rdp: 22
  Foreign:
    Yes: 15
  Gender:
    Fem: 48
    Masc: 2
  Mood:
    Ind: 4
    Nec: 2
  NameType:
    Oth: 3
    Giv: 1
  Num:
    Plur: 7
  NumType:
    Ord: 722
    Mult: 38
    Card: 1
  Number:
    Plur: 1274
    Sing: 11
  Person:
    3: 10
    1: 1
  Poss:
    Yes: 11
  PronType:
    Dem: 4
    Prs: 3
  Reflex:
    Yes: 11
  Style:
    Expr: 1
  Subcat:
    Tran: 441
    Intr: 126
  Tense:
    Past: 394


In [ ]:
# check for doublets in feats

from collections import defaultdict

# ======================================================
# STORAGE
# ======================================================

# Each entry = one problematic token
doublet_report = []

# ======================================================
# SCAN CORPUS
# ======================================================

for sentence in my_corpus:

    text_id = sentence.metadata.get("source_filename", "UNKNOWN_TEXT")
    sent_id = sentence.metadata.get("SENTENCE ID", "UNKNOWN_SENTENCE")

    for token in sentence.get_tokens():

        feats = token.feats
        if not feats:
            continue

        # conllu may normalize duplicates into lists,
        # so we detect list-valued features
        for feat_key, feat_value in feats.items():

            if isinstance(feat_value, list):
                doublet_report.append({
                    "text": text_id,
                    "sentence": sent_id,
                    "form": token.form,
                    "lemma": token.lemma,
                    "upos": token.upos,
                    "feat_key": feat_key,
                    "values": feat_value
                })

# ======================================================
# REPORT
# ======================================================

if not doublet_report:
    print("✔ No duplicate FEATS keys found in corpus.")
else:
    print(f"⚠ Found {len(doublet_report)} tokens with duplicate FEATS keys:\n")

    for item in doublet_report:
        print(f"Text:     {item['text']}")
        print(f"Sentence: {item['sentence']}")
        print(f"Form:     {item['form']}")
        print(f"Lemma:    {item['lemma']}")
        print(f"UPOS:     {item['upos']}")
        print(f"FEAT:     {item['feat_key']} = {item['values']}")
        print("-" * 50)


✔ No duplicate FEATS keys found in corpus.


In [ ]:
#suffix statistics

In [ ]:
from collections import defaultdict, Counter

# ==================================================
# CONFIGURATION
# ==================================================

KNOWN_SUFFIXES = {
    "īh", "īg", "ān", "išn", "tar", "tom", "wand", "war", "ag", "ak",
    "gar", "gār", "kar", "kār", "dār", "tār", "bār", "bed", "bad", "ōmand",
    "garīh", "gārīh", "kārīh", "dārīh", "tārīh", "karīh",
    "dom", "āwand", "awend", "karīgīh", "kārīgīh", "gārīgīh",
    "wandīh", "wendīh", "āwandīh", "padīh", "badīh", "bedīh",
    "kārīg", "gārīg", "tan", "dan", "išnīh", "īhā", "āg", "ār",
    "ēn", "ist", "īd", "ād", "gān", "īdan", "kird", "gird"
}

EXCLUDED_CORES = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār",
    "īg", "ān", "ēn", "man", "pad", "fra", "ni"
}

MIN_CORE_LEN = 3

# ==================================================
# STEP 1: COLLECT TOKENS FROM my_corpus
# ==================================================

all_tokens = []
tokens_by_text = defaultdict(list)

for sentence in my_corpus:
    text_id = sentence.metadata.get("source_filename")
    if not text_id:
        raise ValueError("Sentence without source_filename metadata found!")

    for token in sentence.get_tokens():
        form = str(token.form).strip()
        if form:
            all_tokens.append(form)
            tokens_by_text[text_id].append(form)

print(f"Texts found: {len(tokens_by_text)}")
print(f"Total tokens collected: {len(all_tokens)}")

# ==================================================
# STEP 2: ATTESTED BASES
# ==================================================

attested_bases = {
    t for t in set(all_tokens)
    if " " not in t and len(t) >= MIN_CORE_LEN
}

# ==================================================
# STEP 3: SUFFIX MATCHING SETUP
# ==================================================

sorted_suffixes = sorted(KNOWN_SUFFIXES, key=len, reverse=True)

def valid_core(core):
    return (
        len(core) >= MIN_CORE_LEN
        and core not in EXCLUDED_CORES
        and core in attested_bases
    )

# ==================================================
# STEP 4: COUNTERS
# ==================================================

# Global
global_suffix_tokens = Counter()
global_suffix_types = defaultdict(set)

# Per text
suffix_tokens_by_text = defaultdict(Counter)
suffix_types_by_text = defaultdict(lambda: defaultdict(set))  # 👈 NEW

# ==================================================
# STEP 5: DETECT SUFFIXES
# ==================================================

for text_id, token_list in tokens_by_text.items():
    for token in token_list:
        if " " in token:
            continue

        for suffix in sorted_suffixes:
            if token.endswith(suffix) and len(token) > len(suffix):
                core = token[:-len(suffix)]

                if valid_core(core):
                    # Global
                    global_suffix_tokens[suffix] += 1
                    global_suffix_types[suffix].add(token)

                    # Per text
                    suffix_tokens_by_text[text_id][suffix] += 1
                    suffix_types_by_text[text_id][suffix].add(token)

                break  # longest suffix only

# ==================================================
# STEP 6: GLOBAL RESULTS
# ==================================================

print("\nGLOBAL SUFFIX COUNTS\n")

for suffix in sorted(global_suffix_tokens):
    print(
        f"{suffix:10} "
        f"tokens: {global_suffix_tokens[suffix]:5} | "
        f"types: {len(global_suffix_types[suffix]):4}"
    )

# ==================================================
# STEP 7: PER-TEXT RESULTS (TOKENS + TYPES)
# ==================================================

print("\nSUFFIX COUNTS PER TEXT\n")

for text_id in sorted(suffix_tokens_by_text):
    if not suffix_tokens_by_text[text_id]:
        continue

    print(f"\n{text_id}")
    for suffix in suffix_tokens_by_text[text_id]:
        token_count = suffix_tokens_by_text[text_id][suffix]
        type_count = len(suffix_types_by_text[text_id][suffix])
        print(
            f"  {suffix:10} "
            f"tokens: {token_count:4} | "
            f"types: {type_count:3}"
        )


Texts found: 40
Total tokens collected: 579208

GLOBAL SUFFIX COUNTS

ag         tokens:  9819 | types:  538
ak         tokens:   100 | types:    8
bed        tokens:   222 | types:   13
bedīh      tokens:    23 | types:   10
bār        tokens:   208 | types:    8
dan        tokens:   410 | types:   27
dom        tokens:   110 | types:   15
dār        tokens:   128 | types:   39
dārīh      tokens:   143 | types:   25
gar        tokens:   437 | types:   64
garīh      tokens:   250 | types:   47
gird       tokens:    42 | types:    2
gān        tokens:   224 | types:   35
gār        tokens:   102 | types:   13
gārīg      tokens:     3 | types:    2
gārīh      tokens:   103 | types:   12
ist        tokens:   690 | types:   56
išn        tokens:  9359 | types:  222
išnīh      tokens:   550 | types:  116
kar        tokens:    35 | types:    7
karīh      tokens:     6 | types:    3
kird       tokens:    74 | types:   21
kār        tokens:   143 | types:   23
kārīg      tokens:     8 | types:

In [ ]:
from collections import defaultdict, Counter

# ==================================================
# CONSTRAINTS
# ==================================================

EXCLUDED_P1_SPLIT_LEMMATA = {
    "war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
    "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"
}

EXCLUDED_P2_SUFFIX_LEMMATA = {
    "čand", "dard", "mard", "sad", "ohrmazd", "rōd",
    "fraward", "wimand", "pxrad", "xrafstar",
    "andar", "aštar", "pahikār"
}

suffix_chains = {
    "garīh":   ["gar", "īh"],
    "gārīh":   ["gār", "īh"],
    "karīh":   ["kar", "īh"],
    "kārīh":   ["kār", "īh"],
    "dārīh":   ["dār", "īh"],
    "tārīh":   ["tār", "īh"],
    "bārīh":   ["bār", "īh"],

    "karīgīh": ["kar", "īg", "īh"],
    "kārīgīh": ["kār", "īg", "īh"],
    "garīgīh": ["gar", "īg", "īh"],
    "gārīgīh": ["gār", "īg", "īh"],

    "wandīh":  ["wand", "īh"],
    "padīh":   ["pad", "īh"],
    "išnīh":   ["išn", "īh"],

    "karīg":   ["kar", "īg"],
    "kārīg":   ["kār", "īg"],
    "garīg":   ["gar", "īg"],
    "gārīg":   ["gār", "īg"],

    "bedīh":   ["bed", "īh"],
    "badīh":   ["bad", "īh"],
}

MIN_CORE_LEN = 3

# ==================================================
# STEP 1: COLLECT TOKENS FROM my_corpus
# ==================================================

all_tokens = []
tokens_by_text = defaultdict(list)

for sentence in my_corpus:
    text_id = sentence.metadata.get("source_filename")
    if not text_id:
        raise ValueError("Sentence missing source_filename metadata")

    for token in sentence.get_tokens():
        form = str(token.form).strip()
        if form and " " not in form:
            all_tokens.append(form)
            tokens_by_text[text_id].append(form)

# ==================================================
# STEP 2: ATTESTED BASE LEMMATA
# ==================================================

attested_lemmas = {
    t for t in set(all_tokens)
    if len(t) >= MIN_CORE_LEN
}

# ==================================================
# STEP 3: HELPERS
# ==================================================

def valid_core(core):
    return (
        core in attested_lemmas
        and core not in EXCLUDED_P1_SPLIT_LEMMATA
        and core not in EXCLUDED_P2_SUFFIX_LEMMATA
        and len(core) >= MIN_CORE_LEN
    )

def match_suffix_chain(word, chain):
    remainder = word
    for suffix in reversed(chain):
        if remainder.endswith(suffix):
            remainder = remainder[:-len(suffix)]
        else:
            return None
    return remainder if remainder else None

# ==================================================
# STEP 4: COUNTERS
# ==================================================

# Global
global_chain_tokens = Counter()
global_chain_types = defaultdict(set)

# Per text
per_text_chain_tokens = defaultdict(Counter)
per_text_chain_types = defaultdict(lambda: defaultdict(set))

# ==================================================
# STEP 5: APPLY CHAINS
# ==================================================

for text_id, tokens in tokens_by_text.items():
    for token in tokens:
        for label, chain in suffix_chains.items():
            base = match_suffix_chain(token, chain)
            if base and valid_core(base):
                global_chain_tokens[label] += 1
                global_chain_types[label].add(token)

                per_text_chain_tokens[text_id][label] += 1
                per_text_chain_types[text_id][label].add(token)
                break  # one chain per token

# ==================================================
# STEP 6: GLOBAL REPORT
# ==================================================

print("\nGLOBAL SUFFIX-CHAIN STATISTICS\n")

for label in sorted(global_chain_tokens):
    print(
        f"{label:10} "
        f"tokens: {global_chain_tokens[label]:5} | "
        f"types: {len(global_chain_types[label]):4}"
    )

# ==================================================
# STEP 7: PER-TEXT REPORT (TOKENS + TYPES)
# ==================================================

print("\nSUFFIX-CHAIN COUNTS PER TEXT\n")

for text_id in sorted(per_text_chain_tokens):
    if not per_text_chain_tokens[text_id]:
        continue

    print(f"\n{text_id}")
    for label in per_text_chain_tokens[text_id]:
        token_count = per_text_chain_tokens[text_id][label]
        type_count = len(per_text_chain_types[text_id][label])
        print(f"  {label:10} tokens: {token_count:4} | types: {type_count:3}")



GLOBAL SUFFIX-CHAIN STATISTICS

bedīh      tokens:    23 | types:   10
bārīh      tokens:     6 | types:    3
dārīh      tokens:   143 | types:   25
garīg      tokens:     2 | types:    2
garīgīh    tokens:     1 | types:    1
garīh      tokens:   249 | types:   46
gārīg      tokens:     3 | types:    2
gārīh      tokens:   103 | types:   12
išnīh      tokens:   548 | types:  115
karīh      tokens:     6 | types:    3
kārīg      tokens:     8 | types:    6
kārīgīh    tokens:     2 | types:    2
kārīh      tokens:   382 | types:   53
padīh      tokens:     1 | types:    1
tārīh      tokens:    19 | types:    4
wandīh     tokens:    74 | types:    4

SUFFIX-CHAIN COUNTS PER TEXT


AOD-K20_mptf.conllu
  garīh      tokens:    4 | types:   2
  wandīh     tokens:    1 | types:   1

Col-K7b-2_mptf.conllu
  išnīh      tokens:    1 | types:   1

DD-K35_mptf.conllu
  dārīh      tokens:   15 | types:   5
  išnīh      tokens:   27 | types:  19
  garīh      tokens:   18 | types:   7
  kārīh      t

In [ ]:
#searching for compounds

In [ ]:
from collections import defaultdict
import re

# ------------------------
# CONFIGURATION / FILTERS
# ------------------------
SPLIT_COMPOUND_PART1_DEPRELS = {"nmod", "amod", "compound", "det"}
NOMINAL_HEAD_DEPRELS = {"nsubj", "obj", "obl", "iobj", "vocative", "expl", "dislocated", "root"}
PERMITTED_COMPOUND_PART_UPOS = {"NOUN", "ADJ", "ADV", "VERB"}
EXCLUDED_P1_UPOS = {"PROPN"}
EXCLUDED_P1_SPLIT_LEMMATA = {"war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
                              "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"}
MIN_COMPOUND_PART_LENGTH = 3

# ------------------------
# HELPERS
# ------------------------
def normalize_lemma(lemma):
    """Lowercase, strip spaces, remove digits/special chars if needed."""
    return re.sub(r'[^a-zā-ž0-9]', '', lemma.lower().strip())

def normalize_verbform(feats):
    """Return '-VerbForm' if p2 has a VerbForm, safely handling None."""
    feats = feats or {}  # ensures feats is always a dict
    return "-VerbForm" if feats.get("VerbForm") else ""

# ------------------------
# PRECOLLECT LEMMA -> TOKEN
# ------------------------
all_lemmata = set()
global_lemma_to_token = {}

for sentence in my_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        all_lemmata.add(lemma_norm)
        if lemma_norm not in global_lemma_to_token:
            global_lemma_to_token[lemma_norm] = token  # first occurrence

# ------------------------
# INITIALIZE STORAGE
# ------------------------
compound_examples = []

# ------------------------
# ONE-TOKEN COMPOUNDS (morphological)
# ------------------------
for sentence in my_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        # Skip short tokens
        if len(lemma_norm) < 6:
            continue
        # Try all splits
        # Try all splits
        for i in range(3, len(lemma_norm)-2):
            raw_p1, raw_p2 = lemma_norm[:i], lemma_norm[i:]
            p1, p2 = normalize_lemma(raw_p1), normalize_lemma(raw_p2)  # remove digits/special chars
            if p1 in all_lemmata and p2 in all_lemmata:
                # Fetch p2_token using normalized p2
                p2_token = global_lemma_to_token.get(p2)
                if p2_token is None:
                    continue  # skip if not found
                p1_upos = getattr(global_lemma_to_token.get(p1, None), "upos", "UNK")
                if p1_upos in EXCLUDED_P1_UPOS:
                    continue
                # Determine suffix
                p2_feats = getattr(p2_token, "feats", {}) or {}
                matches = [s for s in KNOWN_SUFFIXES if p2.endswith(s)]  # p2 is normalized
                sfx = max(matches, key=len) if matches else "NoSuffix"
                sfx = suffix_chains.get(sfx.lower(), sfx)
                sfx += normalize_verbform(p2_feats)
                compound_examples.append(
                    (token.lemma, p1, p2_token, "one_token", sfx)
                )
                break  # only first valid split

# ------------------------
# TWO-TOKEN COMPOUNDS (syntactic)
# ------------------------
for sentence_idx, sentence in enumerate(syntactically_parsed_corpus, 1):
    tokens = sentence.get_tokens()
    id_to_token = {t.id: t for t in tokens}
    head_to_children = defaultdict(list)
    for t in tokens:
        head_to_children[getattr(t, 'head', None)].append(t)

    for p1_token in tokens:
        p1_lemma = normalize_lemma(p1_token.lemma)
        p1_upos = p1_token.upos
        # Check filters
        if not (p1_token.deprel in SPLIT_COMPOUND_PART1_DEPRELS and
                p1_upos in PERMITTED_COMPOUND_PART_UPOS and
                len(p1_lemma) >= MIN_COMPOUND_PART_LENGTH and
                p1_lemma not in EXCLUDED_P1_SPLIT_LEMMATA):
            continue

        if p1_token.id in head_to_children and len(head_to_children[p1_token.id]) > 0:
            continue  # skip if p1 has any children

        p2_token = id_to_token.get(getattr(p1_token, "head", None))
        if p2_token is None:
            continue
        p2_lemma = normalize_lemma(p2_token.lemma)
        if not (p2_token.deprel in NOMINAL_HEAD_DEPRELS and
                p2_token.upos in PERMITTED_COMPOUND_PART_UPOS and
                len(p2_lemma) >= MIN_COMPOUND_PART_LENGTH):
            continue

        # Determine suffix with VerbForm
        p2_feats = getattr(p2_token, "feats", {}) or {}
        matches = [s for s in KNOWN_SUFFIXES if p2_lemma.endswith(s)]
        sfx = max(matches, key=len) if matches else "NoSuffix"
        sfx = suffix_chains.get(sfx.lower(), sfx)
        sfx += normalize_verbform(p2_feats)
        suffix = sfx

        compound_examples.append(
            (p1_token.lemma, p1_lemma, p2_token, "two_token", suffix)
        )

print(f"Collected {len(compound_examples)} compounds (one-token + two-token).")


Collected 24109 compounds (one-token + two-token).


In [ ]:
import csv
from collections import defaultdict

# ------------------------
# INITIALIZE FREQUENCY & TEXT/SENTENCE AGGREGATION
# ------------------------
compound_freq = defaultdict(int)
compound_texts = defaultdict(set)
compound_sentences = defaultdict(set)

# --- STEP 1: index compounds by first token for faster matching ---
first_token_to_compounds = defaultdict(list)
for surface_lemma, p1_norm, p2_token, ctype, suffix in compound_examples:
    first_token = normalize_lemma(surface_lemma.split()[0])
    first_token_to_compounds[first_token].append((surface_lemma, ctype, suffix))

# --- STEP 2: iterate over sentences ---
# --- STEP 2: iterate over sentences ---
for sentence in my_corpus:
    sentence_id = sentence.metadata.get("SENTENCE ID", "UNKNOWN_SENTENCE_ID")
    
    # Use source_filename as text ID
    text_id = sentence.metadata.get("source_filename", "UNKNOWN_TEXT_ID")
    
    token_forms = [normalize_lemma(t.lemma) for t in sentence.get_tokens()]

    for i in range(len(token_forms)):
        first_token = token_forms[i]
        if first_token not in first_token_to_compounds:
            continue
        for surface_lemma, ctype, suffix in first_token_to_compounds[first_token]:
            lemma_tokens = [normalize_lemma(t) for t in surface_lemma.split()]
            if i + len(lemma_tokens) > len(token_forms):
                continue
            if token_forms[i:i+len(lemma_tokens)] == lemma_tokens:
                # Count frequency and aggregate sentence/text IDs
                compound_freq[surface_lemma] += 1
                compound_texts[surface_lemma].add(text_id)
                compound_sentences[surface_lemma].add(sentence_id)
                break  # only count first match per sentence

# ------------------------
# WRITE TO CSV
# ------------------------
csv_file_path = "potential_corpus_compounds.csv"

with open(csv_file_path, mode='w', encoding='utf-8-sig', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["Compound", "Frequency", "Texts", "Sentence IDs"])
    writer.writeheader()
    for compound in sorted(compound_freq.keys()):
        writer.writerow({
            "Compound": compound,
            "Frequency": compound_freq[compound],
            "Texts": ";".join(sorted(compound_texts[compound])),
            "Sentence IDs": ";".join(sorted(compound_sentences[compound]))
        })

print(f"CSV written to {csv_file_path}. Total compounds: {len(compound_freq)}")


CSV written to potential_corpus_compounds.csv. Total compounds: 5616


In [ ]:
from collections import defaultdict
import re

# ------------------------
# CONFIGURATION / FILTERS
# ------------------------
SPLIT_COMPOUND_PART1_DEPRELS = {"nmod", "amod", "compound", "det", "advmod", "flat", "apposition", "fixed", "nummod"}
NOMINAL_HEAD_DEPRELS = {"nsubj", "obj", "obl", "iobj", "vocative", "expl", "dislocated", "root"}
PERMITTED_COMPOUND_PART_UPOS = {"NOUN", "ADJ", "ADV"}
# EXCLUDED_P1_UPOS was missing and caused a NameError.
# Define it here. Keep it empty by default (no UPOS excluded),
# or add UPOS tags to exclude, e.g. {"VERB", "AUX", "PRON"} as needed.
EXCLUDED_P1_UPOS = set()
EXCLUDED_P1_SPLIT_LEMMATA = {"war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
                              "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"}
MIN_COMPOUND_PART_LENGTH = 3

# ------------------------
# HELPERS
# ------------------------
def normalize_lemma(lemma):
    """Lowercase, strip spaces, remove digits/special chars if needed."""
    return re.sub(r'[^a-zā-ž0-9]', '', lemma.lower().strip())

def normalize_verbform(feats):
    """Return '-VerbForm' if p2 has a VerbForm, safely handling None."""
    feats = feats or {}  # ensures feats is always a dict
    return "-VerbForm" if feats.get("VerbForm") else ""

# ------------------------
# PRECOLLECT LEMMA -> TOKEN
# ------------------------
all_lemmata = set()
global_lemma_to_token = {}

for sentence in my_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        all_lemmata.add(lemma_norm)
        if lemma_norm not in global_lemma_to_token:
            global_lemma_to_token[lemma_norm] = token  # first occurrence

# ------------------------
# INITIALIZE STORAGE
# ------------------------
compound_examples = []

# ------------------------
# ONE-TOKEN COMPOUNDS (morphological)
# ------------------------
for sentence in syntactically_parsed_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        # Skip short tokens
        if len(lemma_norm) < 6 and token.upos:
            continue
        # Try all splits
        # Try all splits
        for i in range(3, len(lemma_norm)-2):
            raw_p1, raw_p2 = lemma_norm[:i], lemma_norm[i:]
            p1, p2 = normalize_lemma(raw_p1), normalize_lemma(raw_p2)  # remove digits/special chars
            if p1 in all_lemmata and p2 in all_lemmata:
                # Fetch p2_token using normalized p2
                p2_token = global_lemma_to_token.get(p2)
                if p2_token is None:
                    continue  # skip if not found
                p1_upos = getattr(global_lemma_to_token.get(p1, None), "upos", "UNK")
                if p1_upos in EXCLUDED_P1_UPOS:
                    continue
                # Determine suffix
                p2_feats = getattr(p2_token, "feats", {}) or {}
                matches = [s for s in KNOWN_SUFFIXES if p2.endswith(s)]  # p2 is normalized
                sfx = max(matches, key=len) if matches else "NoSuffix"
                sfx = suffix_chains.get(sfx.lower(), sfx)
                sfx += normalize_verbform(p2_feats)
                compound_examples.append(
                    (token.lemma, p1, p2_token, "one_token", sfx)
                )
                break  # only first valid split

# ------------------------
# TWO-TOKEN COMPOUNDS (syntactic)
# ------------------------
def starts_with_any(text, prefixes):
    for prefix in prefixes:
        if text.startswith(prefix):
            return True
    return False

for sentence_idx, sentence in enumerate(syntactically_parsed_corpus, 1):
    tokens = sentence.get_tokens()
    id_to_token = {t.id: t for t in tokens}
    head_to_children = defaultdict(list)
    for t in tokens:
        head_to_children[getattr(t, 'head', None)].append(t)

    for p1_token in tokens:
        p1_lemma = normalize_lemma(p1_token.lemma)
        p1_upos = p1_token.upos
        # Check filters
        if not (starts_with_any(p1_token.deprel, SPLIT_COMPOUND_PART1_DEPRELS) and
                        len(p1_lemma) >= MIN_COMPOUND_PART_LENGTH and
                        p1_lemma not in EXCLUDED_P1_SPLIT_LEMMATA):
                    continue
        if p1_token.id in head_to_children and len(head_to_children[p1_token.id]) > 0:
            continue  # skip if p1 has any children

        p2_token = id_to_token.get(getattr(p1_token, "head", None))
        if p2_token is None:
            continue
        p2_lemma = normalize_lemma(p2_token.lemma)
        if not (starts_with_any(p1_token.deprel, SPLIT_COMPOUND_PART1_DEPRELS) and
                len(p2_lemma) >= MIN_COMPOUND_PART_LENGTH):
            continue

        # Determine suffix with VerbForm
        p2_feats = getattr(p2_token, "feats", {}) or {}
        matches = [s for s in KNOWN_SUFFIXES if p2_lemma.endswith(s)]
        sfx = max(matches, key=len) if matches else "NoSuffix"
        sfx = suffix_chains.get(sfx.lower(), sfx)
        sfx += normalize_verbform(p2_feats)
        suffix = sfx

        compound_examples.append(
            (p1_token.lemma, p1_lemma, p2_token, "two_token", suffix)
        )

print(f"Collected {len(compound_examples)} compounds (one-token + two-token).")



# ------------------------
# FIND OVERLAPS BETWEEN ONE-LEMMA AND TWO-LEMMA COMPOUNDS
# ------------------------

# Extract (p1, p2) pairs for each type, normalized
one_token_pairs = {
    (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    for _, p1, p2_token, compound_type, _ in compound_examples
    if compound_type == "one_token"
}

two_token_pairs = {
    (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    for _, p1, p2_token, compound_type, _ in compound_examples
    if compound_type == "two_token"
}

# Find intersection (overlaps)
overlap_pairs = one_token_pairs & two_token_pairs

from collections import defaultdict

# ------------------------
# Map representative surface forms
# ------------------------
one_token_map = {}
two_token_map = {}

for lemma, p1, p2_token, ctype, _ in compound_examples:
    pair = (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    if pair not in overlap_pairs:
        continue
    if ctype == "one_token" and " " not in lemma:
        one_token_map.setdefault(pair, lemma)
    elif ctype == "two_token":
        two_token_map.setdefault(pair, lemma)

from collections import defaultdict

# ------------------------
# Collect frequencies and sentence IDs separately for one-token and two-token compounds
# ------------------------
from collections import defaultdict

# ------------------------
# Initialize frequency and sentence storage
# ------------------------
pair_to_sentences = defaultdict(set)
pair_to_freq_one = defaultdict(int)
pair_to_freq_two = defaultdict(int)

for sentence in syntactically_parsed_corpus:
    sentence_id = sentence.metadata.get('SENTENCE ID', 'UNKNOWN SENTENCE ID')
    
    tokens = sentence.get_tokens()
    for token in tokens:
        lemma_norm = normalize_lemma(token.lemma)
        
        # Check one-token compounds
        for pair, one_form in one_token_map.items():
            if one_form == token.lemma:
                pair_to_freq_one[pair] += 1
                pair_to_sentences[pair].add(sentence_id)
        
        # Check two-token compounds
        for pair, two_form in two_token_map.items():
            p1, p2 = pair
            # If the two-token form appears (normalized) in the tokens
            token_forms = " ".join(normalize_lemma(t.lemma) for t in tokens)
            if f"{normalize_lemma(two_form)} {p2}" in token_forms:
                pair_to_freq_two[pair] += 1
                pair_to_sentences[pair].add(sentence_id)

# ------------------------
# Print table with separate frequencies
# ------------------------
def clean_sentence_id(sentence_id):
    if sentence_id == 'UNKNOWN SENTENCE ID':
        return sentence_id
    match = re.match(r'(.+?_sentence_\d+)', sentence_id)
    if match:
        return match.group(1)
    return sentence_id.rstrip('_')

print("\nOverlapping Compounds (One-token vs Two-token) with Separate Frequencies:")
print("=" * 140)
print(f"{'One-token':<25} {'Two-token (joined)':<25} {'P1 (norm)':<12} {'P2 (norm)':<12} {'Freq One':<9} {'Freq Two':<9} {'Sentence IDs'}")
print("-" * 140)

for pair in overlap_pairs:
    p1, p2 = pair
    one_form = one_token_map.get(pair, "")
    if not one_form:
        continue
    two_form = two_token_map.get(pair, "")
    joined_two_form = f"{normalize_lemma(two_form)} {p2}" if two_form else ""
    freq_one = pair_to_freq_one.get(pair, 0)
    freq_two = pair_to_freq_two.get(pair, 0)
    sent_ids = ", ".join(clean_sentence_id(sid) for sid in sorted(pair_to_sentences.get(pair, [])))
    print(f"{one_form:<25} {joined_two_form:<25} {p1:<12} {p2:<12} {freq_one:<9} {freq_two:<9} {sent_ids}")

print("=" * 140)
print(f"Total overlapping lemma pairs (one-token without spaces): {len(one_token_map)}")



Collected 22696 compounds (one-token + two-token).

Overlapping Compounds (One-token vs Two-token) with Separate Frequencies:
One-token                 Two-token (joined)        P1 (norm)    P2 (norm)    Freq One  Freq Two  Sentence IDs
--------------------------------------------------------------------------------------------------------------------------------------------
aspzahāy                  asp zahāy                 asp          zahāy        4         56        GBd-TD1 s2142	_	_	_	_	_	_	_	_	, GBd-TD1 s774	_	_	_	_	_	_	_	_	, GBd-TD1 s784	_	_	_	_	_	_	_	_	, GBd-TD1-01_sentence_2143, GBd-TD1-01_sentence_775, GBd-TD1-01_sentence_785
xwēštan                   xwēš tan                  xwēš         tan          46        130       AOD-K20 s24	_	_	_	_	_	_	_	_	, AOD-K20-01_sentence_24, DD-K35 s121	_	_	_	_	_	_	_	_	, DD-K35 s122	_	_	_	_	_	_	_	_	, DD-K35 s309	_	_	_	_	_	_	_	_	, DD-K35 s74	_	_	_	_	_	_	_	_	, DD-K35-01_sentence_121, DD-K35-01_sentence_122, DD-K35-01_sentence_308, DD-K35-01_se

In [ ]:
from collections import defaultdict
import re

# ------------------------
# CONFIGURATION / FILTERS
# ------------------------
SPLIT_COMPOUND_PART1_DEPRELS = {"nmod", "amod", "compound", "det", "advmod", "flat", "apposition", "fixed", "nummod"}
NOMINAL_HEAD_DEPRELS = {"nsubj", "obj", "obl", "iobj", "vocative", "expl", "dislocated", "root"}
PERMITTED_COMPOUND_PART_UPOS = {"NOUN", "ADJ", "ADV"}
EXCLUDED_P1_UPOS = set()
EXCLUDED_P1_SPLIT_LEMMATA = {"war", "wār", "bar", "gar", "gār", "bān", "dār", "īg", "ān",
                              "man", "ēn", "hēr", "pad", "fra", "ni", "ōy"}
MIN_COMPOUND_PART_LENGTH = 3
MAX_COMPOUND_TOKENS = 5  # Maximum length for multi-token compounds

# ------------------------
# HELPERS
# ------------------------
def normalize_lemma(lemma):
    return re.sub(r'[^a-zā-ž0-9]', '', lemma.lower().strip())

def normalize_verbform(feats):
    feats = feats or {}
    return "-VerbForm" if feats.get("VerbForm") else ""

def starts_with_any(text, prefixes):
    return any(text.startswith(prefix) for prefix in prefixes)

# ------------------------
# PRECOLLECT LEMMA -> TOKEN
# ------------------------
all_lemmata = set()
global_lemma_to_token = {}
for sentence in my_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        all_lemmata.add(lemma_norm)
        if lemma_norm not in global_lemma_to_token:
            global_lemma_to_token[lemma_norm] = token

# ------------------------
# INITIALIZE STORAGE
# ------------------------
compound_examples = []

# ------------------------
# ONE-TOKEN COMPOUNDS (morphological)
# ------------------------
for sentence in my_corpus:
    for token in sentence.get_tokens():
        lemma_norm = normalize_lemma(token.lemma)
        if len(lemma_norm) < 6:
            continue
        for i in range(3, len(lemma_norm)-2):
            raw_p1, raw_p2 = lemma_norm[:i], lemma_norm[i:]
            p1, p2 = normalize_lemma(raw_p1), normalize_lemma(raw_p2)
            if p1 in all_lemmata and p2 in all_lemmata:
                p2_token = global_lemma_to_token.get(p2)
                if p2_token is None:
                    continue
                p1_upos = getattr(global_lemma_to_token.get(p1, None), "upos", "UNK")
                if p1_upos in EXCLUDED_P1_UPOS:
                    continue
                p2_feats = getattr(p2_token, "feats", {}) or {}
                sfx = "NoSuffix"
                compound_examples.append(
                    (token.lemma, p1, p2_token, "one_token", sfx)
                )
                break

# ------------------------
# MULTI-TOKEN COMPOUNDS (syntactic, 2..N tokens)
# ------------------------
for sentence in my_corpus:
    tokens = sentence.get_tokens()
    id_to_token = {t.id: t for t in tokens}
    head_to_children = defaultdict(list)
    for t in tokens:
        head_to_children[getattr(t, "head", None)].append(t)

    for start_idx, p1_token in enumerate(tokens):
        p1_lemma = normalize_lemma(p1_token.lemma)
        if not (starts_with_any(p1_token.deprel, SPLIT_COMPOUND_PART1_DEPRELS)
                and len(p1_lemma) >= MIN_COMPOUND_PART_LENGTH
                and p1_lemma not in EXCLUDED_P1_SPLIT_LEMMATA):
            continue

        compound_tokens = [p1_token]
        current_token = p1_token
        for _ in range(1, MAX_COMPOUND_TOKENS):
            next_token = id_to_token.get(getattr(current_token, "head", None))
            if not next_token:
                break
            next_lemma = normalize_lemma(next_token.lemma)
            if len(next_lemma) < MIN_COMPOUND_PART_LENGTH:
                break
            if next_token.upos not in PERMITTED_COMPOUND_PART_UPOS:
                break
            if next_token.id in [t.id for t in compound_tokens]:
                break  # avoid cycles
            compound_tokens.append(next_token)
            current_token = next_token
            if len(compound_tokens) >= 2:
                # Determine suffix (last token)
                last_feats = getattr(compound_tokens[-1], "feats", {}) or {}
                sfx = "NoSuffix" + normalize_verbform(last_feats)
                # Save surface forms
                compound_examples.append(
                    (" ".join(t.lemma for t in compound_tokens),
                     normalize_lemma(compound_tokens[0].lemma),
                     compound_tokens[-1],
                     f"{len(compound_tokens)}_token",
                     sfx)
                )

# ------------------------
# OVERLAPS WITH ONE-TOKEN COMPOUNDS
# ------------------------
one_token_pairs = {
    (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    for _, p1, p2_token, ctype, _ in compound_examples
    if ctype == "one_token"
}

multi_token_pairs = {
    (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    for _, p1, p2_token, ctype, _ in compound_examples
    if ctype.endswith("_token") and ctype != "one_token"
}

overlap_pairs = one_token_pairs & multi_token_pairs

# ------------------------
# MAP REPRESENTATIVE SURFACE FORMS
# ------------------------
one_token_map = {}
multi_token_map = {}
for lemma, p1, p2_token, ctype, _ in compound_examples:
    pair = (normalize_lemma(p1), normalize_lemma(p2_token.lemma))
    if pair not in overlap_pairs:
        continue
    if ctype == "one_token" and " " not in lemma:
        one_token_map.setdefault(pair, lemma)
    elif ctype != "one_token":
        multi_token_map.setdefault(pair, lemma)

# ------------------------
# FREQUENCIES & SENTENCES (separate)
# ------------------------
pair_to_compound_sentences = defaultdict(set)  # one-token sentences
pair_to_mwe_sentences = defaultdict(set)       # multi-token sentences
pair_to_freq_one = defaultdict(int)
pair_to_freq_multi = defaultdict(int)

for sentence in my_corpus:
    sentence_id = sentence.metadata.get('SENTENCE ID', 'UNKNOWN SENTENCE ID')
    tokens = sentence.get_tokens()
    token_forms = [normalize_lemma(t.lemma) for t in tokens]

    # --- One-token ---
    for pair, one_form in one_token_map.items():
        if one_form in [t.lemma for t in tokens]:
            pair_to_freq_one[pair] += 1
            pair_to_compound_sentences[pair].add(sentence_id)

    # --- Multi-token / MWE ---
    for pair, multi_form in multi_token_map.items():
        multi_lemmas = multi_form.split()  # list of normalized tokens
        for i in range(len(token_forms) - len(multi_lemmas) + 1):
            if token_forms[i:i+len(multi_lemmas)] == multi_lemmas:
                pair_to_freq_multi[pair] += 1
                pair_to_mwe_sentences[pair].add(sentence_id)
                break  # only count once per sentence

# ------------------------
# PRINT TABLE
# ------------------------
def clean_sentence_id(sentence_id):
    if sentence_id == 'UNKNOWN SENTENCE ID':
        return sentence_id
    match = re.match(r'(.+?_sentence_\d+)', sentence_id)
    if match:
        return match.group(1)
    return sentence_id.rstrip('_')

print("\nOverlapping Compounds (One-token vs Multi-token) with Separate Frequencies:")
print("=" * 140)

print(f"{'One-token':<25} {'Multi-token':<30} {'P1 (norm)':<12} {'P2 (norm)':<12} {'Freq One':<9} {'Freq Multi':<9} {'Compound Sents':<25} {'MWE Sents':<25}")
print("-" * 180)

for pair in overlap_pairs:
    p1, p2 = pair
    one_form = one_token_map.get(pair, "")
    multi_form = multi_token_map.get(pair, "")
    freq_one = pair_to_freq_one.get(pair, 0)
    freq_multi = pair_to_freq_multi.get(pair, 0)
    compound_sents = ";".join(sorted(pair_to_compound_sentences.get(pair, [])))
    mwe_sents = ";".join(sorted(pair_to_mwe_sentences.get(pair, [])))

    print(f"{one_form:<25} {multi_form:<30} {p1:<12} {p2:<12} {freq_one:<9} {freq_multi:<9} {compound_sents:<25} {mwe_sents:<25}")

print("=" * 140)
print(f"Total overlapping lemma pairs (one-token without spaces): {len(one_token_map)}")



Overlapping Compounds (One-token vs Multi-token) with Separate Frequencies:
One-token                 Multi-token                    P1 (norm)    P2 (norm)    Freq One  Freq Multi Compound Sents            MWE Sents                
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
ahlawfrawahr              ahlaw frawahr                  ahlaw        frawahr      1         101       Dk7-B s124	_	_	_	_	_	_	_	_	_ DD-K35 s542	_	_	_	_	_	_	_	_	_;DD-TD4a s541	_	_	_	_	_	_	_	_	_;Dk3-B s2192	_	_	_	_	_	_	_	_	_;Dk3-B s479	_	_	_	_	_	_	_	_	_;Dk3-B s481	_	_	_	_	_	_	_	_	_;Dk7-B s478	_	_	_	_	_	_	_	_	_;Dk9-B s111	_	_	_	_	_	_	_	_	_;Dk9-B s274	_	_	_	_	_	_	_	_	_;Dk9-B s70	_	_	_	_	_	_	_	_	_;Dk9-B s95	_	_	_	_	_	_	_	_	_;GBd-DH s325	_	_	_	_	_	_	_	_	_;GBd-TD2 s328	_	_	_	_	_	_	_	_	_;P-TD2-01_sentence_455	_	_	_	_	_	_	_	_	_;PVr-K7a s180	_	_	_	_	_	_	_	_	_;PVr-K7a s181	_	_	_	_	_	_	_	_	_

In [ ]:
#fixing the problems in the above cell:

In [ ]:
from collections import defaultdict
import re

# ------------------------
# HELPERS
# ------------------------
def normalize_lemma(lemma):
    if not lemma: return ""
    return re.sub(r'[^a-zā-ž0-9]', '', lemma.lower().strip())

def starts_with_any(text, prefixes):
    if not text: return False
    return any(text.startswith(prefix) for prefix in prefixes)

def get_best_id(sentence):
    """Try various ways to find the sentence ID."""
    # 1. Try common metadata keys
    meta = sentence.metadata
    for key in ['SENTENCE ID', 'sentence_id', 'id', 'ID', 'SENTENCE_ID']:
        if key in meta:
            return str(meta[key])
    # 2. Try the attribute .id
    if hasattr(sentence, 'id') and sentence.id:
        return str(sentence.id)
    return "UNKNOWN_ID"

# ----------------------------------------------
# DATA STRUCTURES
# ----------------------------------------------
# Key: (p1_norm, p2_norm) -> Value: set of (SentenceID, tuple_of_token_ids)
pair_to_one_token_instances = defaultdict(set)
pair_to_multi_token_instances = defaultdict(set)

# To store surface forms
pair_to_one_forms = defaultdict(set)
pair_to_multi_forms = defaultdict(set)

# ------------------------
# PRE-CALCULATE LEMMA LOOKUP
# ------------------------
all_lemmata = set()
for sentence in syntactically_parsed_corpus:
    for token in sentence.get_tokens():
        all_lemmata.add(normalize_lemma(token.lemma))

# ------------------------
# DETECTION PASS
# ------------------------
for sentence in syntactically_parsed_corpus:
    s_id = get_best_id(sentence)
    tokens = sentence.get_tokens()
    id_to_token = {t.id: t for t in tokens}

    # 1. ONE-TOKEN DETECTION
    for token in tokens:
        lemma_norm = normalize_lemma(token.lemma)
        if len(lemma_norm) < 6 or " " in token.lemma: continue
        
        for i in range(3, len(lemma_norm)-2):
            p1, p2 = lemma_norm[:i], lemma_norm[i:]
            if p1 in all_lemmata and p2 in all_lemmata:
                pair = (p1, p2)
                # Count this specific token once
                pair_to_one_token_instances[pair].add((s_id, (token.id,)))
                pair_to_one_forms[pair].add(token.lemma)
                break

    # 2. MULTI-TOKEN DETECTION
    for p1_token in tokens:
        p1_norm = normalize_lemma(p1_token.lemma)
        
        # Filter for valid P1
        if not (starts_with_any(p1_token.deprel, SPLIT_COMPOUND_PART1_DEPRELS)
                and len(p1_norm) >= MIN_COMPOUND_PART_LENGTH
                and p1_norm not in EXCLUDED_P1_SPLIT_LEMMATA):
            continue

        # Trace the head chain
        curr = p1_token
        chain = [p1_token]
        
        # We look up the chain to find a valid P2 head
        for _ in range(MAX_COMPOUND_TOKENS):
            head_id = getattr(curr, "head", None)
            if head_id is None or head_id not in id_to_token:
                break
            
            head_token = id_to_token[head_id]
            head_norm = normalize_lemma(head_token.lemma)
            
            # Avoid cycles
            if head_token.id in [t.id for t in chain]:
                break
            
            chain.append(head_token)
            
            # If this head qualifies as a P2
            if (len(head_norm) >= MIN_COMPOUND_PART_LENGTH and 
                head_token.upos in PERMITTED_COMPOUND_PART_UPOS):
                
                # We found a relationship between p1_token and head_token
                pair = (p1_norm, head_norm)
                
                # Use ONLY the IDs of the two words for the unique hit
                # This prevents over-counting if the path is found via different routes
                hit_key = (s_id, tuple(sorted([p1_token.id, head_token.id])))
                
                pair_to_multi_token_instances[pair].add(hit_key)
                
                # Store surface form (the two words)
                mwe_surface = f"{p1_token.lemma} {head_token.lemma}"
                pair_to_multi_forms[pair].add(mwe_surface)
            
            curr = head_token # Keep climbing the tree

# ------------------------
# FIND OVERLAPS
# ------------------------
overlap_pairs = set(pair_to_one_token_instances.keys()) & set(pair_to_multi_token_instances.keys())

# ------------------------
# PRINT TABLE
# ------------------------
print(f"{'One-token Form':<25} {'Multi-token Form':<35} {'P1':<10} {'P2':<10} {'F1':<5} {'FM':<5} {'Compound IDs'}")
print("-" * 150)

for pair in sorted(overlap_pairs):
    p1_n, p2_n = pair
    
    # Unique instances
    f1 = len(pair_to_one_token_instances[pair])
    fm = len(pair_to_multi_token_instances[pair])
    
    # Unique Sentence IDs
    sents_1 = sorted({inst[0] for inst in pair_to_one_token_instances[pair]})
    sents_m = sorted({inst[0] for inst in pair_to_multi_token_instances[pair]})
    
    form1 = list(pair_to_one_forms[pair])[0] if pair_to_one_forms[pair] else ""
    formm = list(pair_to_multi_forms[pair])[0] if pair_to_multi_forms[pair] else ""

    print(f"{form1:<25} {formm:<35} {p1_n:<10} {p2_n:<10} {f1:<5} {fm:<5} {'; '.join(sents_1)}")
    print(f"{'':<85} MWE IDs: {'; '.join(sents_m)}")
    print("-" * 150)

One-token Form            Multi-token Form                    P1         P2         F1    FM    Compound IDs
------------------------------------------------------------------------------------------------------------------------------------------------------
abārōndād                 abārōn dād                          abārōn     dād        2     2     DD-K35 s175	_	_	_	_	_	_	_	_	_; DD-K35-01_sentence_174	_	_	_	_	_	_	_	_	_
                                                                                      MWE IDs: DMX-L19 s39	_	_	_	_	_	_	_	_	_; DMX-L19-01_sentence_39	_	_	_	_	_	_	_	_	_
------------------------------------------------------------------------------------------------------------------------------------------------------
ahlawfrawahr              ahlaw frawahr                       ahlaw      frawahr    2     14    Dk7-B s124	_	_	_	_	_	_	_	_	_; Dk7-B-01_sentence_124	_	_	_	_	_	_	_	_	_
                                                                                      MW